In [1]:
!pip install clustering-benchmarks -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 38.4 MB/s eta 0:00:00


In [2]:
import clustbench
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

In [3]:
battery_datasets_dict = {
                         'fcps': ['atom', 'chainlink', 'engytime', 'hepta', 'lsun', 'target', 'tetra', 'twodiamonds', 'wingnut'],
                         'g2mg': ['g2mg_4_10', 'g2mg_4_20', 'g2mg_4_30', 'g2mg_4_40', 'g2mg_4_50', 'g2mg_4_60', 'g2mg_4_70', 'g2mg_4_80', 'g2mg_4_90', 'g2mg_8_10', 'g2mg_8_20', 'g2mg_8_30', 'g2mg_8_40', 'g2mg_8_50', 'g2mg_8_60', 'g2mg_8_70', 'g2mg_8_80', 'g2mg_8_90', 'g2mg_16_10', 'g2mg_16_20', 'g2mg_16_30', 'g2mg_16_40', 'g2mg_16_50', 'g2mg_16_60', 'g2mg_16_70', 'g2mg_16_80', 'g2mg_16_90', 'g2mg_32_10', 'g2mg_32_20', 'g2mg_32_30', 'g2mg_32_40', 'g2mg_32_50', 'g2mg_32_60', 'g2mg_32_70', 'g2mg_32_80', 'g2mg_32_90', 'g2mg_64_10', 'g2mg_64_20', 'g2mg_64_30', 'g2mg_64_40', 'g2mg_64_50', 'g2mg_64_60', 'g2mg_64_70', 'g2mg_64_80', 'g2mg_64_90', 'g2mg_128_10', 'g2mg_128_20', 'g2mg_128_30', 'g2mg_128_40', 'g2mg_128_50', 'g2mg_128_60', 'g2mg_128_70', 'g2mg_128_80', 'g2mg_128_90'],
                         'h2mg': ['h2mg_4_10', 'h2mg_4_20', 'h2mg_4_30', 'h2mg_4_40', 'h2mg_4_50', 'h2mg_4_60', 'h2mg_4_70', 'h2mg_4_80', 'h2mg_4_90', 'h2mg_8_10', 'h2mg_8_20', 'h2mg_8_30', 'h2mg_8_40', 'h2mg_8_50', 'h2mg_8_60', 'h2mg_8_70', 'h2mg_8_80', 'h2mg_8_90', 'h2mg_16_10', 'h2mg_16_20', 'h2mg_16_30', 'h2mg_16_40', 'h2mg_16_50', 'h2mg_16_60', 'h2mg_16_70', 'h2mg_16_80', 'h2mg_16_90', 'h2mg_32_10', 'h2mg_32_20', 'h2mg_32_30', 'h2mg_32_40', 'h2mg_32_50', 'h2mg_32_60', 'h2mg_32_70', 'h2mg_32_80', 'h2mg_32_90', 'h2mg_64_10', 'h2mg_64_20', 'h2mg_64_30', 'h2mg_64_40', 'h2mg_64_50', 'h2mg_64_60', 'h2mg_64_70', 'h2mg_64_80', 'h2mg_64_90', 'h2mg_128_10', 'h2mg_128_20', 'h2mg_128_30', 'h2mg_128_40', 'h2mg_128_50', 'h2mg_128_60', 'h2mg_128_70', 'h2mg_128_80', 'h2mg_128_90'],
                         }

## Create autoencoder using pytorch

In [4]:
import torch
import torch.nn as nn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def autoencoder_feat_eng(X, n_embeddings=8):

  X = StandardScaler().fit_transform(X)
  X = torch.tensor(X, dtype=torch.float)

  # --- Autoencoder definition ---
  class Autoencoder(nn.Module):
      def __init__(self, input_dim, latent_dim=n_embeddings):
          super().__init__()
          self.encoder = nn.Sequential(
              nn.Linear(input_dim, 64),
              nn.ReLU(),
              nn.Linear(64, latent_dim)
          )
          self.decoder = nn.Sequential(
              nn.Linear(latent_dim, 64),
              nn.ReLU(),
              nn.Linear(64, input_dim)
          )

      def forward(self, x):
          z = self.encoder(x)
          x_hat = self.decoder(z)
          return x_hat, z

  # --- Train AE ---
  input_dim = X.shape[1] # Dynamically set input_dim
  model = Autoencoder(input_dim)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.MSELoss()

  for epoch in range(100):
      x_hat, z = model(X)
      loss = criterion(x_hat, X)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # --- Cluster latent space ---
  with torch.no_grad():
      latent = model.encoder(X).numpy()

  return latent

## Function get_scores

Get the NCA score of a specific dataset using genie mst algorithm

In [10]:
import sklearn.cluster
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np # Import numpy

def get_scores(battery, dataset, apply_scale=False):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data

  if apply_scale:
    # Scale the data
    #scaler = StandardScaler()
    #X_transformed = scaler.fit_transform(X_transformed)
    X_transformed = clustbench.preprocess_data(X_transformed, random_state=42)

  # https://scikit-learn.org/stable/modules/metrics.html#pairwise-metrics-affinities-and-kernels
  # [‘additive_chi2’, ‘chi2’, ‘linear’, ‘poly’, ‘polynomial’, ‘rbf’, ‘laplacian’, ‘sigmoid’, ‘cosine’]
  m = sklearn.cluster.SpectralClustering(n_clusters=b.n_clusters[0], affinity='rbf', random_state=42)
  results = clustbench.fit_predict_many(m, X_transformed, b.n_clusters[0])
  scores = clustbench.get_score(b.labels[0], results)

  return scores

## Function get_scores_with_autoencoder

Get the NCA score of a specific dataset applying the autoencoder transformation to the data and then using genie mst algorithm

In [6]:
import sklearn.cluster
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_autoencoder(battery, dataset, apply_scale=False, n_embeddings=2):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = autoencoder_feat_eng(b.data, n_embeddings=n_embeddings)

  if apply_scale:
    # Scale the data
    #scaler = StandardScaler()
    #X_transformed = scaler.fit_transform(X_transformed)
    X_transformed = clustbench.preprocess_data(X_transformed, random_state=42)

  m = sklearn.cluster.SpectralClustering(n_clusters=b.n_clusters[0], affinity='poly', random_state=42)
  results = clustbench.fit_predict_many(m, X_transformed, b.n_clusters[0])
  scores = clustbench.get_score(b.labels[0], results)

  return scores

## Execute get_scores on all datasets as baseline

In [11]:
import tqdm
import pandas as pd
columns = ['Battery', 'Dataset', 'Genie NCA Score']
df = pd.DataFrame(columns=columns)
scores_lists = {}
for col in columns:
  scores_lists[col] = []

for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Datasets"):
  for dataset in battery_datasets_dict[battery]:
    scores_lists['Battery'].append(battery)
    scores_lists['Dataset'].append(dataset)
    scores_lists['Genie NCA Score'].append(get_scores(battery, dataset, apply_scale=True))

df_baseline = pd.DataFrame.from_dict(scores_lists)

Processing Datasets:  33%|███▎      | 1/3 [00:06<00:13,  6.64s/it]/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:455: UserWarning: Exited at iteration 697 with accuracies 
[7.32892980e-16 3.28211035e-05 3.00090922e-05]
not reaching the requested tolerance 3.0517578125e-05.
Use iteration 689 instead with accuracy 
1.9949158220911482e-05.

  _, diffusion_map = lobpcg(
/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:455: UserWarning: Exited postprocessing with accuracies 
[8.90831909e-16 2.82578987e-05 3.15895760e-05]
not reaching the requested tolerance 3.0517578125e-05.
  _, diffusion_map = lobpcg(
/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:455: UserWarning: Exited at iteration 422 with accuracies 
[3.68906540e-16 6.32704703e-05 2.81646901e-05]
not reaching the requested tolerance 3.0517578125e-05.
Use iteration 422 instead with accuracy 
3.0478386789702943e-05.

  _, diffusion_map = lo

In [ ]:
df_baseline.style.apply(lambda row:['']*len(row), axis=1)

In [ ]:
from matplotlib import pyplot as plt
df_baseline['Genie NCA Score'].plot(kind='hist', bins=20, title='Genie NCA Score')
plt.gca().spines[['top', 'right',]].set_visible(False)

## Execute get_scores_with_autoencoder on all datasets trying different numbers of embeddings (last hidden layer of the neural network)



In [ ]:
n_embeddings_list = [2,4,8,16,32,64]

scores_lists = {}
for n_embs in tqdm.tqdm(n_embeddings_list, desc="Processing Datasets"):
  for battery in battery_datasets_dict.keys():
    for dataset in battery_datasets_dict[battery]:
      column_name = 'Genie+Autoencoder '+ str(n_embs) +' NCA Score'
      if column_name not in scores_lists:
        scores_lists[column_name] = []
      scores_lists[column_name].append(get_scores_with_autoencoder(battery, dataset, n_embeddings=n_embs))

df_ssnn = pd.DataFrame.from_dict(scores_lists)

In [ ]:
df = pd.concat([df_baseline, df_ssnn], axis=1)

# Final Results

In [ ]:
df['Genie+Autoencoder 2 NCA Score (Difference)'] = df['Genie+Autoencoder 2 NCA Score'] - df['Genie NCA Score']
df['Genie+Autoencoder 4 NCA Score (Difference)'] = df['Genie+Autoencoder 4 NCA Score'] - df['Genie NCA Score']
df['Genie+Autoencoder 8 NCA Score (Difference)'] = df['Genie+Autoencoder 8 NCA Score'] - df['Genie NCA Score']
df['Genie+Autoencoder 16 NCA Score (Difference)'] = df['Genie+Autoencoder 16 NCA Score'] - df['Genie NCA Score']
df['Genie+Autoencoder 32 NCA Score (Difference)'] = df['Genie+Autoencoder 32 NCA Score'] - df['Genie NCA Score']
df['Genie+Autoencoder 64 NCA Score (Difference)'] = df['Genie+Autoencoder 64 NCA Score'] - df['Genie NCA Score']

def highlight_scores(row):
    color = 'yellow'
    if row['Genie NCA Score'] < 0.8 and \
     (row['Genie+Autoencoder 2 NCA Score'] > 0.95 or
      row['Genie+Autoencoder 4 NCA Score'] > 0.95 or
      row['Genie+Autoencoder 8 NCA Score'] > 0.95 or
      row['Genie+Autoencoder 16 NCA Score'] > 0.95 or
      row['Genie+Autoencoder 32 NCA Score'] > 0.95 or
      row['Genie+Autoencoder 64 NCA Score'] > 0.95):
        return [f'background-color: {color}'] * len(row)
    return [''] * len(row)

df[df['Genie NCA Score']<0.8].style.apply(highlight_scores, axis=1)